In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [2]:
price = pd.read_csv("../data/forex_price_data.csv")

price["DateTime"] = pd.to_datetime(price["DateTime"])

In [3]:
price = price.dropna().copy()

In [4]:
price["Target"] = np.where(
    price["Close"].shift(-1) > price["Close"],
    1,
    0
)

price = price.dropna().copy()

In [6]:
features = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Spread",
    "MA_5",
    "MA_20",
    "RSI",
    "MACD",
    "MACD_Signal",
    "BB_Upper",
    "BB_Middle",
    "BB_Lower",
    "Price_Change",
    "High_Low_Range",
    "Return"
]

X = price[features]
y = price["Target"]

KeyError: "['MA_5', 'MA_20', 'RSI', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Middle', 'BB_Lower', 'Price_Change', 'High_Low_Range', 'Return'] not in index"

In [7]:
print(price.columns.tolist())

['DateTime', 'PairID', 'PairName', 'Open', 'High', 'Low', 'Close', 'Volume', 'Spread', 'Target']


In [8]:
from ta.momentum import RSIIndicator
from ta.trend import MACD
from ta.volatility import BollingerBands

In [9]:
# Moving Averages
price["MA_5"] = price["Close"].rolling(window=5).mean()
price["MA_20"] = price["Close"].rolling(window=20).mean()

# Basic Features
price["Price_Change"] = price["Close"] - price["Open"]
price["High_Low_Range"] = price["High"] - price["Low"]
price["Return"] = price["Close"].pct_change()

# RSI
price["RSI"] = RSIIndicator(
    close=price["Close"],
    window=14
).rsi()

# MACD
macd = MACD(close=price["Close"])

price["MACD"] = macd.macd()
price["MACD_Signal"] = macd.macd_signal()

# Bollinger Bands
bb = BollingerBands(
    close=price["Close"],
    window=20,
    window_dev=2
)

price["BB_Upper"] = bb.bollinger_hband()
price["BB_Middle"] = bb.bollinger_mavg()
price["BB_Lower"] = bb.bollinger_lband()

# Remove rows with missing values
price = price.dropna().copy()

In [10]:
print(price.columns.tolist())

['DateTime', 'PairID', 'PairName', 'Open', 'High', 'Low', 'Close', 'Volume', 'Spread', 'Target', 'MA_5', 'MA_20', 'Price_Change', 'High_Low_Range', 'Return', 'RSI', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Middle', 'BB_Lower']


In [11]:
X = price[features]
y = price["Target"]

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Shape:", X_train.shape)
print("Testing Shape :", X_test.shape)

Training Shape: (373, 17)
Testing Shape : (94, 17)


In [13]:
log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train, y_train)

print("✅ Logistic Regression Model Trained")

✅ Logistic Regression Model Trained


In [14]:
log_pred = log_model.predict(X_test)

log_accuracy = accuracy_score(y_test, log_pred)

print(f"Logistic Regression Accuracy: {log_accuracy * 100:.2f}%")

Logistic Regression Accuracy: 52.13%


In [15]:
print(classification_report(y_test, log_pred))

              precision    recall  f1-score   support

           0       0.43      0.51      0.47        39
           1       0.60      0.53      0.56        55

    accuracy                           0.52        94
   macro avg       0.52      0.52      0.52        94
weighted avg       0.53      0.52      0.52        94



In [16]:
from xgboost import XGBClassifier

In [17]:
# Create XGBoost Model

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

# Train Model

xgb_model.fit(X_train, y_train)

print("✅ XGBoost Model Trained Successfully")

✅ XGBoost Model Trained Successfully


In [18]:
# Predict on Test Data

xgb_pred = xgb_model.predict(X_test)

xgb_accuracy = accuracy_score(y_test, xgb_pred)

print(f"XGBoost Accuracy: {xgb_accuracy * 100:.2f}%")

XGBoost Accuracy: 50.00%


In [19]:
print(classification_report(y_test, xgb_pred))

              precision    recall  f1-score   support

           0       0.40      0.41      0.41        39
           1       0.57      0.56      0.57        55

    accuracy                           0.50        94
   macro avg       0.49      0.49      0.49        94
weighted avg       0.50      0.50      0.50        94



In [21]:
# Train Random Forest

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

# Predictions

rf_pred = rf_model.predict(X_test)

# Accuracy

rf_accuracy = accuracy_score(y_test, rf_pred)

print(f"Random Forest Accuracy: {rf_accuracy * 100:.2f}%")

Random Forest Accuracy: 55.32%


In [23]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        log_accuracy,
        rf_accuracy,
        xgb_accuracy
    ]
})

comparison["Accuracy"] = comparison["Accuracy"] * 100

comparison = comparison.sort_values(
    by="Accuracy",
    ascending=False
)

print(comparison)

                 Model   Accuracy
1        Random Forest  55.319149
0  Logistic Regression  52.127660
2              XGBoost  50.000000
